In [1]:
from pathlib import Path
import pandas as pd

In [2]:
folder = Path("target_models")

nodes_output = "raw_nodes.csv"

translation = pd.read_csv("node_names.csv")

edges_output = "raw_edges.csv"

In [3]:
model_map = {
    "033": "BL",
    "034": "HL",
    "035": "BS",
    "036": "HS",
    "037": "SS",
    "038": "SL",
    "140": "Z17",
    "143": "Z21",
    "231": "TGEN",
    "232": "MCF7",
    "233": "T47D",
}

In [4]:
rows = []
index = 1

for aeon_file in sorted(folder.glob("*_inferred.aeon")):
    file_number = aeon_file.stem.split("_")[0]

    model_name = (
        model_map.get(file_number)
        or model_map.get(file_number.lstrip("0"))
        or model_map.get(int(file_number))
    )

    if model_name is None:
        raise KeyError(f"No model mapping for {file_number}")

    nodes = set()
    targets = set()
    edge_count = 0

    with open(aeon_file) as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#") or line.startswith("$"):
                continue

            parts = line.split()
            if len(parts) < 3:
                continue

            regulator = parts[0].removeprefix("v_")
            target = parts[2].removeprefix("v_")

            nodes.add(regulator)
            nodes.add(target)
            targets.add(target)
            edge_count += 1

    input_nodes = nodes - targets

    print(
        f"{aeon_file.name:12s} | "
        f"Model: {model_name:20s} | "
        f"Nodes: {len(nodes):3d} | "
        f"Targets: {len(targets):3d} | "
        f"Inputs: {len(input_nodes):3d} | "
        f"Edges: {edge_count:4d}"
    )

    if input_nodes:
        print("    Input nodes:", ", ".join(sorted(input_nodes)))

    for node in sorted(nodes):
        rows.append({
            "index": index,
            "BBM node": node,
            "Model": model_name,
            "Input in original": "input" if node in input_nodes else ""
        })

        index += 1

df = pd.DataFrame(rows)

print(f"\nProcessed {len(df['Model'].unique())} models.")
print(f"Total rows: {len(df)}")

033_inferred.aeon | Model: BL                   | Nodes:  24 | Targets:  19 | Inputs:   5 | Edges:   68
    Input nodes: Nfkb, erlotinib, pertuzumab, stimulus, trastuzumab
034_inferred.aeon | Model: HL                   | Nodes:  23 | Targets:  19 | Inputs:   4 | Edges:   68
    Input nodes: erlotinib, pertuzumab, stimulus, trastuzumab
035_inferred.aeon | Model: BS                   | Nodes:  16 | Targets:  11 | Inputs:   5 | Edges:   46
    Input nodes: EGF, HRG, erlotinib, pertuzumab, trastuzumab
036_inferred.aeon | Model: HS                   | Nodes:  16 | Targets:  11 | Inputs:   5 | Edges:   46
    Input nodes: EGF, HRG, erlotinib, pertuzumab, trastuzumab
037_inferred.aeon | Model: SS                   | Nodes:  16 | Targets:  11 | Inputs:   5 | Edges:   41
    Input nodes: EGF, HRG, erlotinib, pertuzumab, trastuzumab
038_inferred.aeon | Model: SL                   | Nodes:  25 | Targets:  21 | Inputs:   4 | Edges:   81
    Input nodes: erlotinib, pertuzumab, stimulus, trastuzuma

In [5]:
df

,index,BBM node,Model,Input in original
0,1,AKT,BL,
1,2,CyclinB1,BL,
2,3,CyclinD1,BL,
3,4,ERBB1,BL,
4,5,ERBB2,BL,
...,...,...,...,...
643,644,TRADD,T47D,
644,645,TSC2,T47D,
645,646,TWIST1,T47D,
646,647,WNT1,T47D,input


In [6]:
# Summary statistics across all models
print("\nOverall statistics")
print("------------------")
print(f"Models:            {df['Model'].nunique()}")
print(f"Rows:              {len(df)}")
print(f"Unique node names: {df['BBM node'].nunique()}")

unique_input_nodes = df.loc[df["Input in original"] == "input", "BBM node"].nunique()
print(f"Unique input node names: {unique_input_nodes}")


Overall statistics
------------------
Models:            11
Rows:              648
Unique node names: 229
Unique input node names: 42


In [7]:
if "Our notation" not in translation.columns:
    raise ValueError("Translation table must contain a column named 'Our notation'.")

In [8]:
model_columns = list(model_map.values())

In [9]:
required_columns = {"Our notation", "Input in merge"}
missing = required_columns - set(translation.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [10]:
# mapping[model_name][original_node] = {
#     "Our notation": ...,
#     "input in merge": ...
# }
mapping = {}

for model in model_columns:
    mapping[model] = {}

    for _, row in translation.iterrows():
        node = row[model]

        if pd.isna(node) or str(node).strip() == "":
            continue

        node = str(node).strip()
        our = str(row["Our notation"]).strip()

        input_merge = row["Input in merge"]
        if pd.isna(input_merge):
            input_merge = ""
        else:
            input_merge = str(input_merge).strip()

        if our == "":
            raise ValueError(
                f"Model '{model}', node '{node}' has no corresponding 'Our notation'."
            )

        if node in mapping[model]:
            raise ValueError(
                f"Duplicate node '{node}' in model '{model}'."
            )

        mapping[model][node] = {
            "Our notation": our,
            "Input in merge": input_merge,
        }

print(f"Loaded mappings for {len(mapping)} models.")

for model in model_columns:
    print(f"{model:25s}: {len(mapping[model])} mappings")


Loaded mappings for 11 models.
BL                       : 24 mappings
HL                       : 23 mappings
BS                       : 16 mappings
HS                       : 16 mappings
SS                       : 16 mappings
SL                       : 25 mappings
Z17                      : 80 mappings
Z21                      : 97 mappings
TGEN                     : 117 mappings
MCF7                     : 117 mappings
T47D                     : 117 mappings


In [11]:
# ------------------------------------------------------------------
# Check that every node in the dataframe has a mapping
# ------------------------------------------------------------------

missing = []

for _, row in df.iterrows():
    model = row["Model"]
    node = row["BBM node"]

    if model not in mapping:
        missing.append((model, node, "Model not found in translation table"))
    elif node not in mapping[model]:
        missing.append((model, node, "Node not found"))

if missing:
    missing_df = pd.DataFrame(
        missing,
        columns=["Model", "BBM node", "reason"]
    )

    print("\nMissing mappings:")
    display(missing_df.sort_values(["Model", "BBM node"]))

    raise ValueError(f"{len(missing_df)} nodes are missing a mapping.")

print("\nAll nodes successfully mapped.")


All nodes successfully mapped.


In [12]:
# ------------------------------------------------------------------
# Add unified notation to the dataframe
# ------------------------------------------------------------------

df["Our notation"] = [
    mapping[model][node]["Our notation"]
    for model, node in zip(df["Model"], df["BBM node"])
]

df["Input in merge"] = [
    mapping[model][node]["Input in merge"]
    for model, node in zip(df["Model"], df["BBM node"])
]

display(df.head())

,index,BBM node,Model,Input in original,Our notation,Input in merge
0,1,AKT,BL,,AKT,
1,2,CyclinB1,BL,,CyclinB1,
2,3,CyclinD1,BL,,CyclinD,
3,4,ERBB1,BL,,EGFR,
4,5,ERBB2,BL,,HER2,


In [13]:
# Save to CSV
df.to_csv(nodes_output, index=False)

print(f"\nSaved dataframe to '{nodes_output}'.")


Saved dataframe to 'raw_nodes.csv'.


In [14]:
rows = []
index = 1

for aeon_file in sorted(folder.glob("*_inferred.aeon")):
    file_number = aeon_file.stem.split("_")[0]

    model_name = (
        model_map.get(file_number)
        or model_map.get(file_number.lstrip("0"))
        or model_map.get(int(file_number))
    )

    if model_name is None:
        raise KeyError(f"No model mapping for {file_number}")

    edge_count = 0

    with open(aeon_file) as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#") or line.startswith("$"):
                continue

            parts = line.split()
            if len(parts) != 3:
                continue

            regulator, interaction, target = parts

            regulator = regulator.removeprefix("v_")
            target = target.removeprefix("v_")

            try:
                regulator = mapping[model_name][regulator]["Our notation"]
            except KeyError:
                raise KeyError(
                    f"'{regulator}' in model '{model_name}' has no mapping."
                )

            try:
                target = mapping[model_name][target]["Our notation"]
            except KeyError:
                raise KeyError(
                    f"'{target}' in model '{model_name}' has no mapping."
                )

            if interaction == "->":
                sign = "positive"
            elif interaction == "-|":
                sign = "negative"
            elif interaction == "-?":
                sign = "ambiguous"
            else:
                raise ValueError(
                    f"Unknown interaction '{interaction}' in {aeon_file.name}"
                )

            rows.append({
                "index": index,
                "Regulator": regulator,
                "Target": target,
                "Sign": sign,
                "Model": model_name,
            })

            index += 1
            edge_count += 1

    print(f"{aeon_file.name:12s} | {model_name:25s} | Edges: {edge_count}")

edges_df = pd.DataFrame(rows)

print("\nOverall statistics")
print("------------------")
print(f"Models:           {edges_df['Model'].nunique()}")
print(f"Edges:            {len(edges_df)}")
print(f"Unique regulators:{edges_df['Regulator'].nunique()}")
print(f"Unique targets:   {edges_df['Target'].nunique()}")

edges_df.to_csv(edges_output, index=False)

display(edges_df.head())

033_inferred.aeon | BL                        | Edges: 68
034_inferred.aeon | HL                        | Edges: 68
035_inferred.aeon | BS                        | Edges: 46
036_inferred.aeon | HS                        | Edges: 46
037_inferred.aeon | SS                        | Edges: 41
038_inferred.aeon | SL                        | Edges: 81
140_inferred.aeon | Z17                       | Edges: 211
143_inferred.aeon | Z21                       | Edges: 223
231_inferred.aeon | TGEN                      | Edges: 268
232_inferred.aeon | MCF7                      | Edges: 275
233_inferred.aeon | T47D                      | Edges: 260

Overall statistics
------------------
Models:           11
Edges:            1587
Unique regulators:194
Unique targets:   166


,index,Regulator,Target,Sign,Model
0,1,AKT,AKT,positive,BL
1,2,EGFR,AKT,positive,BL
2,3,HER2,AKT,positive,BL
3,4,HER3,AKT,positive,BL
4,5,PTEN,AKT,negative,BL
